# Notebook 5: Text-to-SQL

Notebook 4 failed on aggregation and cross-table questions. Here the model writes SQL instead of reading retrieved sentences. Every change is scored on questions from `eval/questions.json`: `dev` questions guided the prompt, and `test` questions were fixed beforehand.

In [ ]:
import json
import os
import re
import duckdb
import ollama
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

con = duckdb.connect(str(PROJECT_ROOT / "data" / "f1.db"), read_only=True)
client = ollama.Client(host=os.environ.get("F1_OLLAMA_URL", "http://localhost:11434"))

MODEL = "llama3.2:3b"
# temperature 0 and a fixed seed make runs repeatable, so a score change means the
# prompt or code changed, not the dice.
LLM_OPTIONS = {"temperature": 0, "seed": 42}

with open(PROJECT_ROOT / "eval" / "questions.json", encoding="utf-8") as f:
    QUESTIONS = json.load(f)
DEV = [q for q in QUESTIONS if q["split"] == "dev"]
TEST = [q for q in QUESTIONS if q["split"] == "test"]
TEST2 = [q for q in QUESTIONS if q["split"] == "test2"]

print(f"DuckDB: {con.execute('SELECT COUNT(*) FROM f1_results').fetchone()[0]} rows")
print(f"Ollama: {[m.model for m in client.list().models]}")
print(f"Questions: {len(DEV)} dev, {len(TEST)} test, {len(TEST2)} test2")

## Prompt v1: the failure case

Kept so the before and after can be measured. Three of the six questions first used to test it were also its own few-shot examples, which is why it looked better than it was. It has no JOIN example, no definition of podium, pole or DNF, and it leaves out `f1_drivers`.

In [ ]:
PROMPT_V1 = """You are a DuckDB SQL expert. Use ONLY the table and columns listed.

PRIMARY TABLE — use this for race results, wins, points, positions:
f1_results: season, round, race_name, driver_name, constructor, grid, position, points, status

ONLY use other tables when f1_results cannot answer the question:
f1_circuits: circuit_id, circuit_name, locality, country
f1_races: season, round, race_name, circuit_id, date
f1_qualifying: season, round, driver_id, position, q1, q2, q3
f1_driver_standings: season, position, points, wins, driver_id, constructor
f1_constructor_standings: season, position, points, wins, constructor_id, constructor

RULES:
- Always use f1_results for anything about wins, race positions, or points
- Filter wins with: TRY_CAST(position AS INTEGER) = 1
- Return ONLY raw SQL, nothing else

EXAMPLES:
Q: Who has the most race wins of all time?
A: SELECT driver_name, COUNT(*) AS wins FROM f1_results WHERE TRY_CAST(position AS INTEGER) = 1 GROUP BY driver_name ORDER BY wins DESC LIMIT 10

Q: Who won the 2021 Abu Dhabi Grand Prix?
A: SELECT driver_name, constructor FROM f1_results WHERE season = 2021 AND race_name LIKE '%Abu Dhabi%' AND TRY_CAST(position AS INTEGER) = 1

Q: Most wins for Ferrari in the 2000s?
A: SELECT driver_name, COUNT(*) AS wins FROM f1_results WHERE constructor = 'Ferrari' AND season BETWEEN 2000 AND 2009 AND TRY_CAST(position AS INTEGER) = 1 GROUP BY driver_name ORDER BY wins DESC"""

## Prompt v2

Each addition answers an observed failure:

- Join rules: the model kept inventing `circuit_name` on `f1_results`.
- Value formats: it searched `driver_id = 'Lewis Hamilton'`.
- F1 terms: it invented `status = 'DNF'` and used `COUNT(*)` for points.
- `NO_SQL`: without a refusal path, unanswerable questions crashed or returned something irrelevant.

Pole position is `grid = 1` because `f1_qualifying` starts in 1994, which gives Senna 3 poles instead of 65.

In [ ]:
PROMPT_V2 = """You write DuckDB SQL for a Formula 1 database. Use ONLY these tables and columns.

f1_results (one row per driver per race): season, round, race_name, driver_id, driver_name, constructor, grid, position, points, status
f1_races (one row per race): season, round, race_name, circuit_id, date
f1_circuits: circuit_id, circuit_name, locality, country
f1_drivers: driver_id, given_name, family_name, dob, nationality
f1_driver_standings (one row per driver per season): season, position, points, wins, driver_id, constructor
f1_constructor_standings (one row per team per season): season, position, points, wins, constructor

JOINS:
- f1_results has no circuit columns. For a circuit or country, JOIN f1_races USING (season, round), then JOIN f1_circuits USING (circuit_id).
- For date of birth or nationality, JOIN f1_drivers USING (driver_id).

VALUES:
- driver_name is the full name, like 'Lewis Hamilton'. Filter drivers with driver_name, never driver_id.
- driver_id and circuit_id are lowercase, like 'hamilton' and 'monza'.
- race_name is like 'Monaco Grand Prix'.

F1 TERMS:
- win: position = 1
- podium: position <= 3
- pole position: grid = 1 in f1_results
- did not finish: status <> 'Finished' AND status <> 'Lapped' AND status NOT LIKE '+%'
- total points: SUM(points), never COUNT(*)
- average per season: SUM(points) / COUNT(DISTINCT season)
- championship winner: position = 1 in a standings table

RULES:
- Return one SELECT statement and nothing else.
- If the tables cannot answer the question (weather, tyres, pit stops, lap-by-lap data, opinions), return exactly NO_SQL.

EXAMPLES:
Q: Which driver won the most races in 2012?
A: SELECT driver_name, COUNT(*) AS wins FROM f1_results WHERE season = 2012 AND position = 1 GROUP BY driver_name ORDER BY wins DESC LIMIT 1

Q: How many points did Ferrari score in 2004?
A: SELECT SUM(points) AS total_points FROM f1_results WHERE season = 2004 AND constructor = 'Ferrari'

Q: Which driver won the most races at Monza?
A: SELECT r.driver_name, COUNT(*) AS wins FROM f1_results r JOIN f1_races c USING (season, round) WHERE c.circuit_id = 'monza' AND r.position = 1 GROUP BY r.driver_name ORDER BY wins DESC LIMIT 1

Q: Which team finished second in the 2011 constructors championship?
A: SELECT constructor FROM f1_constructor_standings WHERE season = 2011 AND position = 2

Q: Who was the youngest driver to finish on the podium?
A: SELECT r.driver_name FROM f1_results r JOIN f1_races c USING (season, round) JOIN f1_drivers d USING (driver_id) WHERE r.position <= 3 ORDER BY c.date - d.dob ASC LIMIT 1

Q: What was the track temperature at the 2018 Italian Grand Prix?
A: NO_SQL"""

## Prompt v3: only what the question needs

Prompt v2 scored worse than v1. The model copied its examples, adding `circuit_id = 'monza'` to a career wins question and `season = 2018` to a career poles question. A 3B model doesn't apply a long rule list; it pattern-matches on whatever is nearby.

So Python chooses the context. The base prompt stays short and describes one table, and a regex adds a note and example only when the question needs them. This is a small semantic layer: F1 definitions live in code, not in the model's memory. Questions about data the dataset lacks are refused without calling the model.

In [ ]:
BASE_PROMPT = """Write one DuckDB SELECT statement. Return only the SQL.

f1_results (one row per driver per race): season, round, race_name, driver_id, driver_name, constructor, grid, position, points, status
position = 1 is a win. race_name looks like 'Monaco Grand Prix'.
Use the names and years from the question. Only filter on what the question asks for."""

# (when the question matches, the note to add, an example of the pattern)
HINTS = [
    (r"\bpodiums?\b", "A podium means position <= 3.",
     "Q: How many podiums did Graham Hill score?\nA: SELECT COUNT(*) FROM f1_results WHERE driver_name = 'Graham Hill' AND position <= 3"),
    (r"\bpoles?\b", "A pole position means grid = 1.",
     "Q: How many pole positions did Jim Clark take?\nA: SELECT COUNT(*) FROM f1_results WHERE driver_name = 'Jim Clark' AND grid = 1"),
    (r"\b(not finish|fail(ed)? to finish|retire|dnf)", "Not finishing means status <> 'Finished' AND status <> 'Lapped' AND status NOT LIKE '+%'.",
     "Q: How many races did Jean Alesi fail to finish in 1995?\nA: SELECT COUNT(*) FROM f1_results WHERE driver_name = 'Jean Alesi' AND season = 1995 AND status <> 'Finished' AND status <> 'Lapped' AND status NOT LIKE '+%'"),
    (r"\bper season\b", "An average per season is SUM(points) / COUNT(DISTINCT season).",
     "Q: What is the average points scored by Williams per season?\nA: SELECT SUM(points) / COUNT(DISTINCT season) AS average FROM f1_results WHERE constructor = 'Williams'"),
    (r"\bpoints\b(?!.*per season)", "Total points means SUM(points), not COUNT(*).",
     "Q: Which constructor scored the most points in 1996?\nA: SELECT constructor, SUM(points) AS total FROM f1_results WHERE season = 1996 GROUP BY constructor ORDER BY total DESC LIMIT 1"),
    (r"\bconstructors? championship\b", "Use f1_constructor_standings: season, position, points, wins, constructor.",
     "Q: Which team won the 2011 constructors championship?\nA: SELECT constructor FROM f1_constructor_standings WHERE season = 2011 AND position = 1"),
    (r"\bdrivers? championship\b", "Use f1_driver_standings: season, position, points, wins, driver_id.",
     "Q: Who won the 2011 drivers championship?\nA: SELECT driver_id FROM f1_driver_standings WHERE season = 2011 AND position = 1"),
    (r"\b(oldest|youngest)\b", "Age comes from f1_races.date and f1_drivers.dob. JOIN f1_races USING (season, round) and f1_drivers USING (driver_id).",
     "Q: Who was the youngest driver to finish on the podium?\nA: SELECT r.driver_name FROM f1_results r JOIN f1_races c USING (season, round) JOIN f1_drivers d USING (driver_id) WHERE r.position <= 3 ORDER BY c.date - d.dob ASC LIMIT 1"),
    (r"\bat (?!the\b)[A-Z]|\bcircuit\b", "f1_results has no circuit columns. JOIN f1_races USING (season, round). f1_races.circuit_id is the lowercase circuit name.",
     "Q: Which driver won the most races at Hockenheim?\nA: SELECT r.driver_name, COUNT(*) AS wins FROM f1_results r JOIN f1_races c USING (season, round) WHERE c.circuit_id = 'hockenheimring' AND r.position = 1 GROUP BY r.driver_name ORDER BY wins DESC LIMIT 1"),
]

# The dataset has none of this, so Python refuses without asking the model.
NOT_IN_DATA = re.compile(r"\b(weather|rain|rainfall|temperature|tyres?|tires?|pit stops?|lap times?|favou?rite|popular|opinions?)\b", re.IGNORECASE)


def build_prompt_v3(question: str):
    """Returns the prompt, or None when the question needs data we don't have."""
    if NOT_IN_DATA.search(question):
        return None
    matched = [(note, example) for pattern, note, example in HINTS
               if re.search(pattern, question, flags=0 if "[A-Z]" in pattern else re.IGNORECASE)]
    notes = "\n".join(note for note, _ in matched)
    examples = "\n\n".join(example for _, example in matched)
    return "\n\n".join(part for part in (BASE_PROMPT, notes, examples) if part)


print(build_prompt_v3("How many pole positions does Lewis Hamilton have?"))

In [ ]:
NO_SQL = "NO_SQL"

def clean_sql(text: str) -> str:
    """Pull one statement out of the model's reply. Multi-line SQL is kept."""
    text = text.replace("```sql", "").replace("```", "").strip()
    match = re.search(r"\b(SELECT|WITH)\b", text, flags=re.IGNORECASE)
    if match is None:
        return NO_SQL  # an English refusal is a refusal, not a parser error
    return text[match.start():].split(";")[0].split("\n\n")[0].strip()


def resolve_prompt(prompt, question: str):
    """A prompt is either fixed text or a function that builds one for the question."""
    return prompt(question) if callable(prompt) else prompt


def generate_sql(question: str, prompt=build_prompt_v3) -> str:
    text = resolve_prompt(prompt, question)
    if text is None:
        return NO_SQL
    response = client.chat(
        model=MODEL,
        messages=[{"role": "user", "content": f"{text}\n\nQ: {question}\nA:"}],
        options=LLM_OPTIONS,
    )
    return clean_sql(response["message"]["content"])

In [ ]:
def execute_sql(sql: str) -> tuple:
    """
    Execute SQL against DuckDB. Returns (dataframe, error).
    """
    try:
        result = con.execute(sql).df()
        return result, None
    except Exception as e:
        return None, str(e)

## Retry on database errors

A retry can only fix queries that fail. A query that runs and returns the wrong thing gives no signal, which is why the evaluation counts `silent_wrong` separately.

In [ ]:
def fix_sql(question: str, sql: str, error: str, prompt=build_prompt_v3) -> str:
    """Show the model its own failed query and DuckDB's error message."""
    message = f"""{resolve_prompt(prompt, question)}

The query below failed. Return one corrected SELECT statement and nothing else.

Q: {question}
Failed SQL: {sql}
DuckDB error: {error.splitlines()[0]}
A:"""
    response = client.chat(
        model=MODEL,
        messages=[{"role": "user", "content": message}],
        options=LLM_OPTIONS,
    )
    return clean_sql(response["message"]["content"])


def run_sql(question: str, prompt=build_prompt_v3, max_retries: int = 2) -> dict:
    """Generate and execute, feeding errors back up to max_retries times."""
    sql = generate_sql(question, prompt)
    attempts = 1
    while True:
        if sql == NO_SQL:
            return {"sql": sql, "df": None, "error": None, "attempts": attempts}
        df, error = execute_sql(sql)
        if error is None or attempts > max_retries:
            return {"sql": sql, "df": df, "error": error, "attempts": attempts}
        sql = fix_sql(question, sql, error, prompt)
        attempts += 1

## Answering from the result

The model turned a correct query into "38 wins" when the data said 16, so Python checks the sentence: every number in the answer must appear in the data or the question.

In [ ]:
def numbers_in(text: str) -> set:
    return {float(n.replace(",", "")) for n in re.findall(r"\d[\d,]*(?:\.\d+)?", text)}


def sql_answer(question: str, result_df) -> str:
    """
    Turn the result into a sentence. Python handles empty results, and any answer
    containing a number the data doesn't contain is replaced by the raw rows.
    """
    if result_df is None or result_df.empty:
        return "The query returned no rows, so I can't answer that."

    result_text = result_df.head(20).to_string(index=False)
    prompt = f"""Answer the question in one sentence using only the data below.
Use the numbers exactly as shown. Do not add any fact or number that is not in the data.

Question: {question}

Data:
{result_text}

Answer:"""

    response = client.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options=LLM_OPTIONS,
    )
    answer = response["message"]["content"].strip()

    allowed = numbers_in(result_text) | numbers_in(question)
    if not numbers_in(answer) <= allowed:
        return f"Result of the query:\n{result_text}"
    return answer

In [ ]:
def ask_sql(question: str, verbose: bool = True) -> str:
    """
    Full Text-to-SQL pipeline:
    1. Generate SQL from question, retrying on database errors
    2. Execute against DuckDB
    3. Generate natural language answer
    """
    if verbose:
        print(f"Question: {question}")

    run = run_sql(question)
    if verbose:
        print(f"\nGenerated SQL ({run['attempts']} attempt(s)):\n{run['sql']}")

    if run["sql"] == NO_SQL:
        answer = "That isn't in the database."
    elif run["error"]:
        answer = f"Could not execute query: {run['error'].splitlines()[0]}"
    else:
        if verbose:
            print(f"\nResult:\n{run['df'].head(10)}\n")
        answer = sql_answer(question, run["df"])

    if verbose:
        print(f"\nAnswer:\n{answer}")
    return answer

In [ ]:
questions = [
    "Who has the most race wins of all time?",
    "Which constructor won the most races in 2022?",
    "Who won the 2021 Abu Dhabi Grand Prix?",  # This failed in Notebook 4
    "Who had the most wins for Ferrari in the 2000s?",  # This also failed
    "Which driver has the most podiums at Monaco?",
    "What is the average points scored by Mercedes per season?",
]

for q in questions:
    print("="*60)
    ask_sql(q)
    print()

## Evaluation

Executing without an error isn't success, so each result is compared with the answer from hand-written SQL.

In [ ]:
def expected_values(q: dict) -> list:
    """Accepted answers: the truth query's value, plus any listed alternatives."""
    if q["truth_sql"] is None:
        return []
    return [con.execute(q["truth_sql"]).fetchone()[0]] + q.get("accept", [])


def same(cell, value) -> bool:
    if isinstance(value, str):
        return isinstance(cell, str) and cell.strip().lower() == value.lower()
    try:
        return abs(float(cell) - float(value)) <= 0.01
    except (TypeError, ValueError):
        return False


def score(q: dict, run: dict) -> str:
    """correct, silent_wrong (ran but wrong), error, or refused (NO_SQL on an answerable question)."""
    if q["truth_sql"] is None:
        return "correct" if run["sql"] == NO_SQL else "silent_wrong"
    if run["sql"] == NO_SQL:
        return "refused"
    if run["error"]:
        return "error"
    if run["df"].empty:
        return "silent_wrong"
    row = list(run["df"].iloc[0])
    hit = any(same(cell, value) for cell in row for value in expected_values(q))
    return "correct" if hit else "silent_wrong"


def evaluate(questions: list, prompt, max_retries: int, label: str) -> dict:
    rows = []
    for q in questions:
        run = run_sql(q["question"], prompt, max_retries)
        rows.append({"id": q["id"], "kind": q["kind"], "result": score(q, run),
                     "attempts": run["attempts"], "sql": run["sql"]})
    totals = {k: sum(r["result"] == k for r in rows) for k in ("correct", "silent_wrong", "error", "refused")}
    print(f"{label:<32} {totals['correct']:>2}/{len(rows)} correct   {totals}")
    return {"label": label, "totals": totals, "rows": rows}

In [ ]:
results = [
    evaluate(DEV, PROMPT_V1, 0, "dev: prompt v1"),
    evaluate(DEV, PROMPT_V2, 0, "dev: prompt v2"),
    evaluate(DEV, PROMPT_V2, 2, "dev: prompt v2 + retry"),
    evaluate(DEV, build_prompt_v3, 0, "dev: prompt v3"),
    evaluate(DEV, build_prompt_v3, 2, "dev: prompt v3 + retry"),
]

In [ ]:
# `test` was seen while judging v2, so only `test2` is a clean score for v3.
results.append(evaluate(TEST, PROMPT_V1, 0, "test: prompt v1"))
results.append(evaluate(TEST, PROMPT_V2, 2, "test: prompt v2 + retry"))
results.append(evaluate(TEST, build_prompt_v3, 2, "test (seen): prompt v3 + retry"))
results.append(evaluate(TEST2, PROMPT_V1, 0, "test2: prompt v1"))
results.append(evaluate(TEST2, build_prompt_v3, 2, "test2: prompt v3 + retry"))

for r in results[-1]["rows"]:
    print(f"\n[{r['result']}] {r['id']} ({r['kind']}, {r['attempts']} attempt(s))\n  {r['sql']}")

out = PROJECT_ROOT / "eval" / "results_nb05.json"
out.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

## SQL path against vector path

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

# Unload the LLM first: on a machine short of memory the embedding model can't load beside it
client.generate(model=MODEL, keep_alive=0)

# Load vector RAG components
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path=str(PROJECT_ROOT / "chroma_db"))
collection = chroma_client.get_collection("f1_corpus")

def ask_vector(question: str) -> str:
    query_vec = embedding_model.encode([question]).tolist()
    results = collection.query(query_embeddings=query_vec, n_results=5)
    context = "\n".join([f"- {doc}" for doc in results["documents"][0]])

    prompt = f"""Answer using ONLY this context. Be concise.
If the context does not state the answer, reply exactly: Not in the context.

Context:
{context}

Question: {question}
Answer:"""

    response = client.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options=LLM_OPTIONS,
    )
    return response["message"]["content"].strip()

In [ ]:
REFUSAL = re.compile(r"(not|isn't) in the (context|database)|no information|can't answer", re.IGNORECASE)

def text_is_right(q: dict, answer: str) -> bool:
    """Both paths end in text, so the head-to-head scores the final answer."""
    if q["truth_sql"] is None:
        return bool(REFUSAL.search(answer))
    for value in expected_values(q):
        if isinstance(value, str) and value.lower() in answer.lower():
            return True
        if not isinstance(value, str) and any(abs(n - float(value)) <= 0.01 for n in numbers_in(answer)):
            return True
    return False


head_to_head = []
for q in TEST2:
    sql_text, vector_text = ask_sql(q["question"], verbose=False), ask_vector(q["question"])
    head_to_head.append({"id": q["id"], "kind": q["kind"], "sql_right": text_is_right(q, sql_text),
                         "vector_right": text_is_right(q, vector_text),
                         "sql_answer": sql_text, "vector_answer": vector_text})

print(f"SQL path:    {sum(r['sql_right'] for r in head_to_head)}/{len(TEST2)}")
print(f"Vector path: {sum(r['vector_right'] for r in head_to_head)}/{len(TEST2)}")
for r in head_to_head:
    print(f"{r['id']} {r['kind']:<14} sql={'Y' if r['sql_right'] else '-'} vector={'Y' if r['vector_right'] else '-'}")

out = PROJECT_ROOT / "eval" / "head_to_head_nb05.json"
out.write_text(json.dumps(head_to_head, indent=2, ensure_ascii=False), encoding="utf-8")

In [ ]:
# Not scored: there is no single right answer, so read these side by side.
for q in ["How did Max Verstappen perform at Monaco?", "Tell me about Ferrari in 2004."]:
    print("="*60)
    print(f"QUESTION: {q}")
    print(f"\nSQL PATH:    {ask_sql(q, verbose=False)}")
    print(f"\nVECTOR PATH: {ask_vector(q)}")
    print()

## What We Learned

All runs use `temperature=0` on `llama3.2:3b`. Scores count the query result, compared with hand-written SQL.

| Question set | v1 (original) | v2 (long prompt) + retry | v3 (selected hints) + retry |
|---|---|---|---|
| dev (12) | 6 | 5 | 7 |
| test (14) | 4 | 2 | 7 |
| test2 (13) | 4 | not run | 10 |

**The scores for v3 are optimistic.**

- `test` had already been seen while judging v2, so its 7/14 is not a clean score.
- `test2` was written before the hints, but by the same author, who knew its questions. `popular` is in the refusal list and test2_13 contains it.
- test2_04 passed by luck: it asked about McLaren, and the SQL returned 1988's top team, which was McLaren.
- Dropping those two leaves 8/13. Across all three sets, v3 is right on 24 of 39 questions. Treat it as a 60-70% system.
- 39 questions is a small sample, and each was run once.

### Findings

1. **More prompt made things worse.** v2 added rules and examples and lost points. The model copied example values, putting `circuit_id = 'monza'` in a career wins question.
2. **Selecting context beat adding context.** v3's short prompt with only matching hints removed most of that copying, but not all of it: test2_01 took `'Monaco Grand Prix'` from the base prompt, and test2_03 used a constructor example for a driver question.
3. **Retry added nothing.** v3 scored 7/12 on dev with and without it. Most failures run without error, so there is nothing for a retry to act on.
4. **Silent failures are the main risk.** v3's 3 wrong test2 results all ran cleanly. test2_01 told the user Nigel Mansell won no races, when the answer is 31.
5. **Python guards work where the model doesn't.** The refusal check and the number check are deterministic.

### SQL path against vector path

On the 13 `test2` questions, scoring the final text: SQL 10, vector 4. The same caveats apply to the SQL score. Both refused the two unanswerable questions, so the gap comes from questions with one exact answer, including counts across many rows that five retrieved sentences can't hold.

Vector scores can shift when the Chroma index is rebuilt, because retrieval can change: test2_09 passed on one index and failed on the next.

The two narrative questions are unscored. For Verstappen at Monaco, SQL returned no rows and vector gave a usable answer, though from only five races. That is two examples, not evidence that vector is better at narrative questions.

### For Notebook 6

Route by question shape, send counts and rankings to SQL, and keep Python deciding definitions. A larger model is worth testing on these same questions once the machine has room for it.
